# 02 - Encoding (TF-IDF Bigrama)

Este notebook toma los parquets preprocesados y construye la codificación **TF-IDF con bigramas**.

## Objetivo
- Convertir `text_clean` en una matriz numérica sparse TF-IDF.
- Evitar **data leakage**: el vocabulario se aprende solo con train.
- Crear un conjunto de **validación** desde train para ajustar hiperparámetros sin tocar test.

## Diferencia respecto a BoW
| Aspecto | BoW (anterior) | TF-IDF Bigrama (este notebook) |
|---|---|---|
| Vectorizador | `CountVectorizer` | `TfidfVectorizer` |
| Ponderación | Frecuencia cruda | TF × IDF (penaliza palabras muy comunes) |
| n-gramas | `(1,1)` unigramas | `(1,2)` uni + bigramas |
| Ejemplo | `"good"` | `"good"`, `"not good"`, `"very happy"` |

## Política de conjuntos
| Conjunto | Tamaño | Uso |
|---|---|---|
| train | 1,360,000 | Entrenar y crear val |
| val | 20% del train | Ajuste de hiperparámetros |
| test | 240,000 | Evaluación final única en 03_baseline.ipynb |

## Reproducibilidad
Se fija `SEED = 42` para que todos los experimentos sean replicables.


## 1. Instalación e imports

In [ ]:
!pip install -q scikit-learn pyarrow joblib scipy

In [ ]:
import pandas as pd
import numpy as np
import joblib
from scipy.sparse import save_npz, load_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

SEED = 42
print('OK - Imports listos. SEED:', SEED)


## 2. Carga de parquets preprocesados

In [ ]:
df_train = pd.read_parquet('sentiment_preprocessed_train.parquet')
df_test  = pd.read_parquet('sentiment_preprocessed_test.parquet')

print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)
df_train.head()


### 2.1 Validaciones previas

Verificamos nulos, columnas requeridas y textos vacíos antes de vectorizar.


In [ ]:
required_cols = {'text_clean', 'label'}
assert required_cols.issubset(df_train.columns), 'Train: faltan columnas requeridas'
assert required_cols.issubset(df_test.columns),  'Test: faltan columnas requeridas'

df_train['text_clean'] = df_train['text_clean'].fillna('')
df_test['text_clean']  = df_test['text_clean'].fillna('')

print('Nulos train:', df_train.isnull().sum().to_dict())
print('Nulos test: ', df_test.isnull().sum().to_dict())
print('Textos vacíos train:', (df_train['text_clean'] == '').sum())
print('Textos vacíos test: ', (df_test['text_clean']  == '').sum())
print('Balance labels train:', df_train['label'].value_counts().to_dict())
print('Balance labels test: ', df_test['label'].value_counts().to_dict())


## 3. TF-IDF con bigramas

### ¿Qué es TF-IDF?
**TF (Term Frequency):** cuántas veces aparece una palabra en el documento.
**IDF (Inverse Document Frequency):** penaliza palabras que aparecen en casi todos los documentos (ej. "the", "is").
El producto TF × IDF da más peso a palabras discriminativas y menos a las muy comunes.

### ¿Por qué bigramas?
Con `ngram_range=(1,2)` se incluyen pares de palabras consecutivas.
Esto captura frases de sentimiento que unigramas no detectan:
- `"not"` solo no dice nada, pero `"not good"` sí.
- `"very happy"` tiene más peso que `"very"` y `"happy"` por separado.

### Regla crítica: evitar data leakage
- `fit_transform(train)`: aprende vocabulario **solo** con train.
- `transform(test)`: aplica el vocabulario aprendido, nunca aprende nada nuevo.

### Parámetros del vectorizador
| Parámetro | Valor | Significado |
|---|---|---|
| `max_features` | 100,000 | Top 100k n-gramas más frecuentes (más que BoW por los bigramas) |
| `min_df` | 5 | Ignorar n-gramas en menos de 5 tweets |
| `max_df` | 0.95 | Ignorar n-gramas en más del 95% de tweets |
| `ngram_range` | (1,2) | Unigramas Y bigramas |
| `sublinear_tf` | True | Aplica `1 + log(tf)` para suavizar frecuencias altas |


In [ ]:
X_raw_train = df_train['text_clean'].values
y_train      = df_train['label'].values

X_raw_test  = df_test['text_clean'].values
y_test       = df_test['label'].values

print(f"Textos en train: {len(X_raw_train):,}")
print(f"Textos en test:  {len(X_raw_test):,}")
print(f"Distribución labels train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Distribución labels test:  {dict(zip(*np.unique(y_test,  return_counts=True)))}")


In [ ]:
# Definir TF-IDF con bigramas — fit SOLO sobre train
vectorizer = TfidfVectorizer(
    max_features=100000,  # más features para cubrir bigramas
    min_df=5,
    max_df=0.95,
    ngram_range=(1, 2),   # unigramas + bigramas
    sublinear_tf=True     # suaviza TF con log(1+tf)
)

X_train_tfidf = vectorizer.fit_transform(X_raw_train)
X_test_tfidf  = vectorizer.transform(X_raw_test)

print(f"Shape X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape X_test_tfidf:  {X_test_tfidf.shape}")
print(f"Tipo de matriz:      {type(X_train_tfidf)}")
print(f"Vocabulario final:   {len(vectorizer.vocabulary_):,} n-gramas")


## 4. Diagnóstico del TF-IDF

Top n-gramas por peso total acumulado y sparsity de la matriz.


In [ ]:
# Top 20 n-gramas con mayor peso acumulado
freq  = X_train_tfidf.sum(axis=0).A1
words = vectorizer.get_feature_names_out()
top20 = pd.Series(freq, index=words).sort_values(ascending=False).head(20)
print('Top 20 n-gramas (mayor peso TF-IDF acumulado):\n')
print(top20.to_string())


In [ ]:
# Sparsity
total      = X_train_tfidf.shape[0] * X_train_tfidf.shape[1]
nonzero    = X_train_tfidf.nnz
sparsity   = (1 - nonzero / total) * 100

print(f"Total elementos posibles: {total:,}")
print(f"Elementos no-cero:        {nonzero:,}")
print(f"Sparsity:                 {sparsity:.2f}%")


## 5. Split de validación (desde train)

Dividimos `X_train_tfidf` en 80/20 para tener un conjunto de validación
sin tocar `X_test_tfidf`.
> `stratify=y_train` garantiza balance 50/50 de labels en ambos splits.


In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_tfidf,
    y_train,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train
)

print('X_tr  (80% train):', X_tr.shape)
print('X_val (20% train):', X_val.shape)
print('X_test (intocado):', X_test_tfidf.shape)
print()
print('Balance y_tr :', dict(zip(*np.unique(y_tr,  return_counts=True))))
print('Balance y_val:', dict(zip(*np.unique(y_val, return_counts=True))))


## 6. Guardar artefactos

| Archivo | Descripción |
|---|---|
| `X_tr.npz` | Train 80% para desarrollo |
| `X_val.npz` | Validación 20% para ajustar hiperparámetros |
| `X_train_tfidf.npz` | Train completo para entrenamiento final |
| `X_test_tfidf.npz` | Test reservado para evaluación final |
| `y_tr/val/train/test.pkl` | Labels correspondientes |
| `tfidf_vectorizer.pkl` | Vectorizador ajustado (necesario para inferencia) |


In [ ]:
save_npz('X_tr.npz',          X_tr)
save_npz('X_val.npz',         X_val)
save_npz('X_train_tfidf.npz', X_train_tfidf)
save_npz('X_test_tfidf.npz',  X_test_tfidf)

joblib.dump(y_tr,    'y_tr.pkl')
joblib.dump(y_val,   'y_val.pkl')
joblib.dump(y_train, 'y_train.pkl')
joblib.dump(y_test,  'y_test.pkl')

joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("Artefactos guardados:")
print("  X_tr.npz              -> train 80% para CV/validación")
print("  X_val.npz             -> validación 20% para hiperparámetros")
print("  X_train_tfidf.npz     -> train completo para entrenamiento final")
print("  X_test_tfidf.npz      -> test SOLO para evaluación final")
print("  y_tr/val/train/test   -> labels correspondientes")
print("  tfidf_vectorizer.pkl  -> vectorizador TF-IDF ajustado")


## 7. Verificación final

In [ ]:
print('Verificación de artefactos guardados:')
print('  X_tr:          ', load_npz('X_tr.npz').shape)
print('  X_val:         ', load_npz('X_val.npz').shape)
print('  X_train_tfidf: ', load_npz('X_train_tfidf.npz').shape)
print('  X_test_tfidf:  ', load_npz('X_test_tfidf.npz').shape)
print(f"  Vocabulario:    {len(joblib.load('tfidf_vectorizer.pkl').vocabulary_):,} n-gramas")
print()
print('Listo para continuar con 03_baseline.ipynb')
